In [0]:
spark.conf.set(
  "fs.azure.account.key.databasedepiset.dfs.core.windows.net",
  "Mn4OtOTNuiBqpUbVUCv2MdLvh1R1QoT+FOe57UVLFcE0/nZb/imc9DbziepKkjb2WZhFI5Lc8pB5+AStoAJTzQ=="
)

In [0]:
df_transaction = spark.read.parquet("abfss://product@databasedepiset.dfs.core.windows.net/transaction/dbo.transactions.parquet")
 
df_transaction.show()

+--------------+-----------+----------+--------+--------+----------------+
|transaction_id|customer_id|product_id|store_id|quantity|transaction_date|
+--------------+-----------+----------+--------+--------+----------------+
|            31|        101|         1|       1|       3|      2025-04-01|
|            32|        102|         2|       2|       2|      2025-04-03|
|            33|        103|         3|       3|       1|      2025-04-05|
|            34|        104|         4|       4|       5|      2025-04-07|
|            35|        105|         5|       5|       2|      2025-04-09|
|            36|        106|         6|       1|       4|      2025-04-11|
|            37|        107|         7|       2|       1|      2025-04-13|
|            38|        108|         8|       3|       3|      2025-04-15|
|            39|        109|         9|       4|       2|      2025-04-17|
|            40|        110|        10|       5|       5|      2025-04-19|
|            41|        1

In [0]:
df_store = spark.read.parquet("abfss://product@databasedepiset.dfs.core.windows.net/store/dbo.stores.parquet")
 
df_store.show()

+--------+--------------------+------------+
|store_id|          store_name|    location|
+--------+--------------------+------------+
|       1|     City Mall Store|         UAE|
|       2|   High Street Store|Saudi Arabia|
|       3|   Tech World Outlet|       Qatar|
|       4|Cairo Festival Ci...|       Egypt|
|       5|          Mega Plaza|      Kuwait|
+--------+--------------------+------------+



In [0]:
df_product = spark.read.parquet("abfss://product@databasedepiset.dfs.core.windows.net/product/dbo.products.parquet")
 
df_product.show()

+----------+-----------------+-----------+-----+
|product_id|     product_name|   category|price|
+----------+-----------------+-----------+-----+
|         1|   Wireless Mouse|Electronics|  800|
|         2|Bluetooth Speaker|Electronics| 1200|
|         3|         Yoga Mat|    Fitness|  499|
|         4|     Laptop Stand|Accessories|  999|
|         5|     Notebook Set| Stationery|  149|
|         6|     Water Bottle|    Fitness|  299|
|         7|       Smartwatch|Electronics| 4999|
|         8|   Desk Organizer|Accessories|  399|
|         9|     Dumbbell Set|    Fitness| 1999|
|        10|   Pen Drive 32GB|Electronics|  599|
+----------+-----------------+-----------+-----+



In [0]:
df_customer = spark.read.parquet("abfss://product@databasedepiset.dfs.core.windows.net/customer/MohammedHameds/Retails_Project/refs/heads/main/dataset/customers.parquet")
 
 
df_customer.show()


+-----------+----------------+--------------------+------------+-----------------+
|customer_id|       full_name|               email|     country|registration_date|
+-----------+----------------+--------------------+------------+-----------------+
|        101|    Ahmed Khaled|ahmed.khaled1@gma...|       Egypt|       2025-10-16|
|        102| Sara Al Mansour|sara.almansour2@o...|Saudi Arabia|       2025-10-18|
|        103|    Layla Kazemi|layla.kazemi3@yah...|         UAE|       2025-10-19|
|        104|     Omar Farouk|omar.farouk4@gmai...|       Egypt|       2025-10-17|
|        105|Fatima Al Rashid|fatima.alrashid5@...|Saudi Arabia|       2025-10-15|
|        106|   Yousef Nasser|yousef.nasser6@gm...|         UAE|       2025-10-16|
|        107|     Mona Hossam|mona.hossam7@yaho...|       Egypt|       2025-10-18|
|        108|  Hassan Al Saud|hassan.alsaud8@ou...|Saudi Arabia|       2025-10-19|
|        109|     Ayesha Khan|ayesha.khan9@gmai...|         UAE|       2025-10-15|
|   

In [0]:
df_transaction.write.mode('overwrite').saveAsTable('depiomar.sales.transactions_bronze')
df_customer.write.mode('overwrite').saveAsTable('depiomar.sales.customers_bronze')
df_store.write.mode('overwrite').saveAsTable('depiomar.sales.stores_bronze')
df_product.write.mode('overwrite').saveAsTable('depiomar.sales.products_bronze')
 

In [0]:
from pyspark.sql.functions import col

In [0]:
df_customer = df_customer.select(
    col("customer_id").cast("int"),
    col("full_name"),
    col("email"),
    col("country"),
    col("registration_date").cast("date")
)

In [0]:
df_retails_silver= df_transaction.join(df_customer,"customer_id").join(df_product,"product_id").join(df_store,"store_id").withColumn("total_amount",col("quantity")*col("price"))
display(df_retails_silver)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,full_name,email,country,registration_date,product_name,category,price,store_name,location,total_amount
3,8,101,58,3,2025-05-25,Ahmed Khaled,ahmed.khaled1@gmail.com,Egypt,2025-10-16,Desk Organizer,Accessories,399,Tech World Outlet,Qatar,1197
4,9,102,59,5,2025-05-27,Sara Al Mansour,sara.almansour2@outlook.com,Saudi Arabia,2025-10-18,Dumbbell Set,Fitness,1999,Cairo Festival City Mall,Egypt,9995
5,10,103,60,1,2025-05-29,Layla Kazemi,layla.kazemi3@yahoo.com,UAE,2025-10-19,Pen Drive 32GB,Electronics,599,Mega Plaza,Kuwait,599
1,1,104,61,2,2025-06-01,Omar Farouk,omar.farouk4@gmail.com,Egypt,2025-10-17,Wireless Mouse,Electronics,800,City Mall Store,UAE,1600
2,2,105,62,3,2025-06-03,Fatima Al Rashid,fatima.alrashid5@hotmail.com,Saudi Arabia,2025-10-15,Bluetooth Speaker,Electronics,1200,High Street Store,Saudi Arabia,3600
3,3,106,63,4,2025-06-05,Yousef Nasser,yousef.nasser6@gmail.com,UAE,2025-10-16,Yoga Mat,Fitness,499,Tech World Outlet,Qatar,1996
4,4,107,64,5,2025-06-07,Mona Hossam,mona.hossam7@yahoo.com,Egypt,2025-10-18,Laptop Stand,Accessories,999,Cairo Festival City Mall,Egypt,4995
5,5,108,65,1,2025-06-09,Hassan Al Saud,hassan.alsaud8@outlook.com,Saudi Arabia,2025-10-19,Notebook Set,Stationery,149,Mega Plaza,Kuwait,149
1,6,109,66,2,2025-06-11,Ayesha Khan,ayesha.khan9@gmail.com,UAE,2025-10-15,Water Bottle,Fitness,299,City Mall Store,UAE,598
2,7,110,67,3,2025-06-13,Tamer Adel,tamer.adel10@hotmail.com,Egypt,2025-10-17,Smartwatch,Electronics,4999,High Street Store,Saudi Arabia,14997


In [0]:
df_retails_silver.write.mode("overwrite").option("mergeSchema","true").saveAsTable('depiomar.sales.transactions_silver')

In [0]:
from pyspark.sql.functions import sum, col
df_top_products=df_retails_silver.groupBy("product_name").agg(sum(col("total_amount")).alias("total_amount"))
df_top_products.write.mode("overwrite").option("mergeSchema","true").saveAsTable('depiomar.sales.products_gold')


In [0]:
df_summary=df_retails_silver.groupBy("product_name","product_id","country","store_id","store_name","location").agg(sum(col("total_amount")).alias("total_amount"),sum(col("quantity")).alias("avg_quantity"))
df_summary.show()
df_summary.write.mode("overwrite").saveAsTable('depiomar.sales.summary_gold')

+-----------------+----------+------------+--------+--------------------+------------+------------+------------+
|     product_name|product_id|     country|store_id|          store_name|    location|total_amount|avg_quantity|
+-----------------+----------+------------+--------+--------------------+------------+------------+------------+
|       Smartwatch|         7|         UAE|       2|   High Street Store|Saudi Arabia|        9998|           2|
|       Smartwatch|         7|Saudi Arabia|       2|   High Street Store|Saudi Arabia|       39992|           8|
|   Desk Organizer|         8|Saudi Arabia|       3|   Tech World Outlet|       Qatar|        2793|           7|
|     Laptop Stand|         4|Saudi Arabia|       4|Cairo Festival Ci...|       Egypt|        6993|           7|
|Bluetooth Speaker|         2|Saudi Arabia|       2|   High Street Store|Saudi Arabia|        6000|           5|
|         Yoga Mat|         3|Saudi Arabia|       3|   Tech World Outlet|       Qatar|         9